# Time Series Analysis — Buckeye / LP Sub-Portfolio
## Beginning Outstanding Balances

This notebook performs a full time-series diagnostic study for the **Buckeye – LP** sub-portfolio.

All analyses are executed twice:
- **Part A — Training data only** (2023-05 → 2024-12-31, ~19 monthly observations)
- **Part B — Full series** (Training + Test period 2025-01 onwards)

This dual-pass approach reveals what the model can "see" at forecast time versus what the full truth looks like.

| Section | Analysis |
|---------|----------|
| 1 | Autocorrelation (ACF) & Partial Autocorrelation (PACF) |
| 2 | Stationarity Tests (ADF, KPSS) + Long/Short Memory (Hurst Exponent) |
| 3.1 | Trend Component — Moving Average & Polynomial Fitting |
| 3.2 | Seasonality — ACF, FFT, STL Decomposition |
| 3.3 | Cycle Detection — Spectral Analysis & Wavelet Transform |
| 4 | Summary Dashboard |

In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring seaborn statsmodels
!pip install --proxy=192.168.2.10:8080 PyWavelets hurst scipy

In [ ]:
import os
import json
import warnings
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import snowflake.connector

# Time-series statistics
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.seasonal import STL

# Spectral / signal
from scipy.fft import rfft, rfftfreq
from scipy.signal import periodogram, find_peaks
import pywt

# Memory / Hurst
from hurst import compute_Hc

warnings.filterwarnings('ignore')

# ── Global plot style ─────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams.update({
    'figure.dpi'      : 120,
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 11,
    'xtick.labelsize' : 9,
    'ytick.labelsize' : 9,
    'legend.fontsize' : 9,
})

PALETTE = {
    'raw'    : '#2196F3',
    'ma3'    : '#FF9800',
    'ma5'    : '#4CAF50',
    'poly1'  : '#9C27B0',
    'poly2'  : '#E91E63',
    'poly3'  : '#009688',
    'actual' : '#F44336',
}

print("All imports successful.")

# Setting Up the Connector

In [ ]:
# load the credentials
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']

In [ ]:
ctx = snowflake.connector.connect(**snowflake_creds)
cur = ctx.cursor()

# Load & Prepare Data

In [ ]:
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df = cur.fetch_pandas_all()
print(f"Total rows loaded: {len(df):,}")
df.head()

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────
PORTFOLIO       = 'Buckeye'
SUB_PORTFOLIO   = 'LP'
TARGET_METRIC   = 'Beginning Outstanding Balances'
LAST_TRAIN_DATE = '2024-12-31'

# ── Filter to Buckeye / LP / Actual ──────────────────────────────────────
portfolio_df = df[
    (df['PORTFOLIO']     == PORTFOLIO) &
    (df['SUB_PORTFOLIO'] == SUB_PORTFOLIO) &
    (df['FORECAST_TYPE'] == 'Actual')
].copy()

print(f"Available metrics for {PORTFOLIO} / {SUB_PORTFOLIO}:")
for m in sorted(portfolio_df['METRIC'].unique()):
    print(f"  • {m}")

# ── Aggregate and pivot ───────────────────────────────────────────────────
dff = portfolio_df.groupby(['DATE', 'METRIC'])['METRIC_VALUE'].sum().reset_index()
data_df = dff.pivot(index='DATE', columns='METRIC', values='METRIC_VALUE').sort_index()

# ── Train / Full split ────────────────────────────────────────────────────
cutoff      = datetime.date.fromisoformat(LAST_TRAIN_DATE)
train_series = data_df.loc[data_df.index <= cutoff, TARGET_METRIC].dropna()
full_series  = data_df[TARGET_METRIC].dropna()

print(f"\n{'─'*55}")
print(f"  Series             : {TARGET_METRIC}")
print(f"  Training points    : {len(train_series)}  ({train_series.index[0]} → {train_series.index[-1]})")
print(f"  Full-series points : {len(full_series)}  ({full_series.index[0]} → {full_series.index[-1]})")
print(f"  Min / Max / Mean (train): {train_series.min():,.0f} / {train_series.max():,.0f} / {train_series.mean():,.0f}")
print(f"{'─'*55}")

In [ ]:
# ── Raw series overview ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=False)

for ax, series, label, color in zip(
    axes,
    [train_series, full_series],
    ['[PART A — TRAIN ONLY]', '[PART B — TRAIN + TEST]'],
    [PALETTE['raw'], PALETTE['actual']],
):
    ax.plot(pd.to_datetime(series.index), series.values,
            marker='o', linewidth=2, color=color, markersize=5, label='Actual')
    ax.axvline(pd.to_datetime(LAST_TRAIN_DATE), color='grey',
               linestyle='--', linewidth=1.2, label='Train/Test cutoff')
    ax.fill_between(pd.to_datetime(series.index), series.values, alpha=0.12, color=color)
    ax.set_title(f"{label}  —  {TARGET_METRIC}  ({PORTFOLIO} / {SUB_PORTFOLIO})")
    ax.set_ylabel('Outstanding Balances ($)')
    ax.set_xlabel('Date')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    # annotate count
    ax.annotate(f"n = {len(series)}", xy=(0.01, 0.92),
                xycoords='axes fraction', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.suptitle('Raw Time Series Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Section 1 — Autocorrelation (ACF) & Partial Autocorrelation (PACF)

**ACF** (AutoCorrelation Function) measures how correlated the series is with its own lagged values — useful for identifying the **MA(q)** order.  
**PACF** (Partial ACF) removes indirect lag effects — useful for identifying the **AR(p)** order.

Significant bars **outside the shaded ±1.96/√n confidence band** indicate statistically meaningful autocorrelation at that lag.

| Pattern | Interpretation |
|---------|---------------|
| ACF decays slowly | Non-stationary / long-memory trend |
| ACF cuts off at lag q | MA(q) component |
| PACF cuts off at lag p | AR(p) component |
| Both taper slowly | ARMA(p,q) |
| Spike at lag 3 or 6 | Quarterly or semi-annual seasonality |

In [ ]:
def plot_acf_pacf(series, label, color_bar='#2196F3', color_sig='#FF9800'):
    """
    Plot ACF and PACF side-by-side with significance band annotation.
    Also prints a plain-text interpretation.
    """
    n    = len(series)
    lags = min(n // 2, 15)
    conf = 1.96 / np.sqrt(n)   # ±1.96/√n significance threshold

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))
    fig.suptitle(f'ACF & PACF  —  {label}  (n={n})',
                 fontsize=14, fontweight='bold')

    for ax, func, title, method in zip(
        axes,
        [plot_acf, plot_pacf],
        ['ACF (MA order indicator)', 'PACF (AR order indicator)'],
        [None, 'ywmle'],
    ):
        kwargs = dict(ax=ax, lags=lags, alpha=0.05,
                      color=color_bar, vlines_props={'colors': color_bar})
        if method:
            kwargs['method'] = method
        func(series.values, **kwargs)

        # Re-colour significance band
        for line in ax.get_lines():
            line.set_color(color_sig)
            line.set_linewidth(0.8)
            line.set_linestyle('--')

        # Annotate quarterly / semi-annual lags
        for lag_mark, lag_label in [(3, 'Q'), (6, 'SA'), (12, 'A')]:
            if lag_mark <= lags:
                ax.axvline(lag_mark, color='#E91E63', linewidth=1.2,
                           linestyle=':', alpha=0.7)
                ax.text(lag_mark + 0.1, ax.get_ylim()[1] * 0.93,
                        lag_label, color='#E91E63', fontsize=8)

        ax.set_title(title)
        ax.set_xlabel('Lag (months)')
        ax.set_ylabel('Correlation')
        ax.axhline(0, color='black', linewidth=0.5)

    plt.tight_layout()
    plt.show()

    # ── Interpretation ──────────────────────────────────────────────────
    acf_vals  = acf(series.values, nlags=lags, fft=True)
    sig_lags  = [i for i, v in enumerate(acf_vals[1:], 1) if abs(v) > conf]

    print(f"\n  Significance threshold (±1.96/√n): ±{conf:.3f}")
    print(f"  Significant ACF lags              : {sig_lags}")
    if 3 in sig_lags:
        print("  → Spike at lag 3 detected — quarterly seasonality likely")
    if 6 in sig_lags:
        print("  → Spike at lag 6 detected — semi-annual seasonality likely")
    if len(sig_lags) >= 4 and max(sig_lags) >= 6:
        print("  → Slow ACF decay — series may be non-stationary (trend present)")
    print()


for series, label in [(train_series, '[PART A — TRAIN ONLY]'),
                      (full_series,  '[PART B — TRAIN + TEST]')]:
    plot_acf_pacf(series, label)

# Section 2 — Stationarity Tests & Long/Short Memory Analysis

## 2.1  Augmented Dickey-Fuller (ADF) Test
- **H₀:** Series has a unit root (non-stationary)  
- Reject H₀ when p-value < 0.05 → **stationary**

## 2.2  KPSS Test (Kwiatkowski-Phillips-Schmidt-Shin)
- **H₀:** Series is stationary around a deterministic trend  
- Reject H₀ when p-value < 0.05 → **non-stationary**

> Using both ADF and KPSS together resolves ambiguity: if ADF fails to reject AND KPSS rejects → strong evidence of unit root (difference-stationary)

## 2.3  Hurst Exponent (H)
| H value | Interpretation |
|---------|---------------|
| H < 0.5 | Mean-reverting / Anti-persistent (short memory) |
| H ≈ 0.5 | Random walk (no memory) |
| H > 0.5 | Trend-persistent (long memory) |

In [ ]:
def stationarity_report(series, label):
    """
    Run ADF, KPSS and Hurst tests; display styled result table + rolling
    mean/std plot to visually inspect stationarity.
    """
    vals = series.values.astype(float)
    n    = len(vals)

    # ── ADF ──────────────────────────────────────────────────────────────
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(vals, autolag='AIC')
    adf_stat_1pct = adf_crit['1%']
    adf_concl = 'Stationary ✓' if adf_p < 0.05 else 'Non-Stationary ✗'

    # ── KPSS ─────────────────────────────────────────────────────────────
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(vals, regression='ct', nlags='auto')
    kpss_concl = 'Non-Stationary ✗' if kpss_p < 0.05 else 'Stationary ✓'

    # ── Hurst ─────────────────────────────────────────────────────────────
    # Need at least 20 points; fall back gracefully
    if n >= 12:
        H, c, _ = compute_Hc(vals, kind='price', simplified=True)
        if H < 0.45:
            hurst_concl = 'Mean-Reverting (short memory)'
        elif H > 0.55:
            hurst_concl = 'Trend-Persistent (long memory)'
        else:
            hurst_concl = 'Random Walk'
    else:
        H          = float('nan')
        hurst_concl = 'Insufficient data (< 12 points)'

    # ── Combined verdict ──────────────────────────────────────────────────
    if adf_p >= 0.05 and kpss_p < 0.05:
        combined = 'UNIT ROOT  — difference-stationary'
        row_color = '#ffcccc'
    elif adf_p < 0.05 and kpss_p >= 0.05:
        combined = 'STATIONARY  — no differencing needed'
        row_color = '#ccffcc'
    else:
        combined = 'AMBIGUOUS  — check trend-stationarity'
        row_color = '#fff3cc'

    # ── Summary table ─────────────────────────────────────────────────────
    summary = pd.DataFrame([
        ['ADF test statistic', f'{adf_stat:.4f}', f'(1% crit: {adf_stat_1pct:.4f})',
         f'p = {adf_p:.4f}', adf_concl],
        ['KPSS test statistic', f'{kpss_stat:.4f}', f'(lags: {kpss_lags})',
         f'p ≈ {kpss_p:.4f}', kpss_concl],
        ['Hurst exponent (H)', f'{H:.4f}' if not np.isnan(H) else 'N/A',
         '0=anti-persist, 0.5=random, 1=persist', '', hurst_concl],
        ['Combined verdict', '', '', '', combined],
    ], columns=['Test', 'Statistic', 'Note', 'p-value', 'Conclusion'])

    print(f"\n{'═'*65}")
    print(f"  STATIONARITY REPORT  —  {label}")
    print(f"{'═'*65}")
    display(summary.style
            .set_properties(**{'text-align': 'left'})
            .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
            .apply(lambda row: [
                f'background-color: {row_color}' if row.name == 3 else ''
                for _ in row], axis=1))

    # ── Rolling mean & std plot ───────────────────────────────────────────
    window = max(3, n // 5)
    roll_mean = series.rolling(window).mean()
    roll_std  = series.rolling(window).std()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 6), sharex=True)
    fig.suptitle(f'Rolling Statistics  —  {label}  (window={window}m)',
                 fontsize=13, fontweight='bold')

    dates = pd.to_datetime(series.index)
    ax1.plot(dates, series.values, color=PALETTE['raw'], label='Raw', linewidth=1.8)
    ax1.plot(dates, roll_mean.values, color=PALETTE['ma3'],
             label=f'Rolling Mean ({window}m)', linewidth=2)
    ax1.axvline(pd.to_datetime(LAST_TRAIN_DATE), color='grey',
                linestyle='--', linewidth=1, label='Cutoff')
    ax1.set_ylabel('Value'); ax1.legend()
    ax1.set_title(f'ADF p={adf_p:.3f}  |  KPSS p≈{kpss_p:.3f}  |  {combined}')

    ax2.plot(dates, roll_std.values, color=PALETTE['poly2'],
             label=f'Rolling Std ({window}m)', linewidth=2)
    ax2.set_ylabel('Std Dev'); ax2.legend()
    ax2.set_xlabel('Date')
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()


for series, label in [(train_series, '[PART A — TRAIN ONLY]'),
                      (full_series,  '[PART B — TRAIN + TEST]')]:
    stationarity_report(series, label)

# Section 3 — Component Analysis

## 3.1  Trend Component

### Moving Average
A centered moving average smooths the series by averaging over a sliding window, removing short-term fluctuations to reveal the underlying trend direction.

- **3-month MA** — captures local trend, sensitive to recent changes
- **5-month MA** — smoother, less reactive to short-term noise

### Polynomial Trend Fitting
Fitting a polynomial of degree 1 (linear), 2 (quadratic) and 3 (cubic) on a numeric time index reveals whether the trend is accelerating, decelerating or changing direction.

A higher R² indicates a better fit, but higher degrees can overfit sparse series.

In [ ]:
def plot_trend(series, label):
    """
    Plot Moving Averages and Polynomial fits to visualise the trend.
    """
    dates = pd.to_datetime(series.index)
    vals  = series.values.astype(float)
    t     = np.arange(len(vals))       # numeric time index for poly fitting

    # ── MA windows ──────────────────────────────────────────────────────
    ma3 = series.rolling(3, center=True).mean()
    ma5 = series.rolling(5, center=True).mean()

    # ── Polynomial fits ──────────────────────────────────────────────────
    poly_fits, poly_r2 = {}, {}
    for deg, key, color in [(1, 'poly1', PALETTE['poly1']),
                            (2, 'poly2', PALETTE['poly2']),
                            (3, 'poly3', PALETTE['poly3'])]:
        coefs = np.polyfit(t, vals, deg)
        fitted = np.polyval(coefs, t)
        ss_res = np.sum((vals - fitted) ** 2)
        ss_tot = np.sum((vals - vals.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0
        poly_fits[key]  = fitted
        poly_r2[key]    = r2

    # ── Figure ────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 9))
    fig.suptitle(f'Trend Component  —  {label}', fontsize=14, fontweight='bold')

    # ── Panel 1: Moving averages ──────────────────────────────────────────
    ax1.plot(dates, vals, color=PALETTE['raw'], linewidth=2,
             marker='o', markersize=4, label='Raw', zorder=3)
    ax1.plot(dates, ma3.values, color=PALETTE['ma3'],
             linewidth=2.5, label='3-Month Moving Avg', linestyle='-')
    ax1.plot(dates, ma5.values, color=PALETTE['ma5'],
             linewidth=2.5, label='5-Month Moving Avg', linestyle='--')

    # Uncertainty band between MA3 and MA5
    valid = ~(np.isnan(ma3.values) | np.isnan(ma5.values))
    ax1.fill_between(dates[valid], ma3.values[valid], ma5.values[valid],
                     alpha=0.2, color='grey', label='MA3–MA5 band')

    ax1.axvline(pd.to_datetime(LAST_TRAIN_DATE), color='grey',
                linestyle=':', linewidth=1.2, label='Train cutoff')
    ax1.set_ylabel('Outstanding Balances ($)')
    ax1.legend(ncol=3)
    ax1.set_title('Moving Average Trend Smoothing')

    # ── Panel 2: Polynomial fits ──────────────────────────────────────────
    ax2.plot(dates, vals, color=PALETTE['raw'], linewidth=2,
             marker='o', markersize=4, label='Raw', zorder=3)

    poly_colors = {'poly1': PALETTE['poly1'], 'poly2': PALETTE['poly2'],
                   'poly3': PALETTE['poly3']}
    poly_labels = {'poly1': 'Degree 1 (linear)',
                   'poly2': 'Degree 2 (quadratic)',
                   'poly3': 'Degree 3 (cubic)'}
    best_key = max(poly_r2, key=poly_r2.get)
    for key in ['poly1', 'poly2', 'poly3']:
        lw = 3 if key == best_key else 1.8
        ls = '-' if key == best_key else '--'
        r2_str = f'R²={poly_r2[key]:.3f}'
        marker = '  ← best fit' if key == best_key else ''
        ax2.plot(dates, poly_fits[key], color=poly_colors[key],
                 linewidth=lw, linestyle=ls,
                 label=f'{poly_labels[key]}  {r2_str}{marker}')

    ax2.axvline(pd.to_datetime(LAST_TRAIN_DATE), color='grey',
                linestyle=':', linewidth=1.2, label='Train cutoff')
    ax2.set_ylabel('Outstanding Balances ($)')
    ax2.set_xlabel('Date')
    ax2.tick_params(axis='x', rotation=45)
    ax2.legend(ncol=2)
    ax2.set_title('Polynomial Trend Fitting')

    plt.tight_layout()
    plt.show()

    # ── Print R² summary ──────────────────────────────────────────────────
    print(f"\n  Polynomial R² Summary:")
    for deg_name, key in [('Degree 1 (linear)', 'poly1'),
                          ('Degree 2 (quadratic)', 'poly2'),
                          ('Degree 3 (cubic)', 'poly3')]:
        marker = '  ← BEST' if key == best_key else ''
        print(f"    {deg_name:<28}: R² = {poly_r2[key]:.4f}{marker}")
    print()


for series, label in [(train_series, '[PART A — TRAIN ONLY]'),
                      (full_series,  '[PART B — TRAIN + TEST]')]:
    plot_trend(series, label)

## 3.2  Seasonality Analysis

Three complementary methods are used to detect periodicity:

| Method | What it detects |
|--------|----------------|
| **ACF Seasonal Plot** | Periodic spikes at lag 3, 6, 12 |
| **FFT (Fast Fourier Transform)** | Dominant frequency / period in the signal; model-free |
| **STL Decomposition** | Trend + Seasonal + Residual split; seasonal strength index |

A **Seasonal Strength Index** near 1.0 means seasonality dominates; near 0 means the residuals carry most variance (weak seasonality).

In [ ]:
def plot_seasonality(series, label):
    """
    3-panel seasonality analysis:
      1. ACF with seasonal lag markers
      2. FFT power spectrum with top-3 dominant periods labelled
      3. STL decomposition (period=3, i.e. quarterly)
    """
    vals  = series.values.astype(float)
    n     = len(vals)
    dates = pd.to_datetime(series.index)
    lags  = min(n // 2, 15)

    # ══════════════════════════════════════════════════════════════════════
    # Panel 1  —  ACF with seasonal lag highlights
    # ══════════════════════════════════════════════════════════════════════
    fig1, ax = plt.subplots(figsize=(14, 4))
    plot_acf(vals, ax=ax, lags=lags, alpha=0.05, color='#2196F3',
             vlines_props={'colors': '#2196F3'})
    for lag_m, period_name, clr in [(3, 'Quarterly (3m)', '#E91E63'),
                                    (6, 'Semi-annual (6m)', '#FF9800'),
                                    (12, 'Annual (12m)', '#4CAF50')]:
        if lag_m <= lags:
            ax.axvline(lag_m, color=clr, linewidth=2, linestyle='--', alpha=0.8)
            ax.text(lag_m + 0.15, ax.get_ylim()[1] * 0.90,
                    period_name, color=clr, fontsize=8, fontweight='bold')
    ax.set_title(f'ACF with Seasonal Lag Markers  —  {label}')
    ax.set_xlabel('Lag (months)'); ax.set_ylabel('ACF')
    plt.tight_layout(); plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # Panel 2  —  FFT Power Spectrum
    # ══════════════════════════════════════════════════════════════════════
    # Detrend first (remove mean) for cleaner spectral analysis
    vals_detr = vals - np.mean(vals)
    fft_vals  = rfft(vals_detr)
    fft_freq  = rfftfreq(n, d=1)     # d=1 month sampling
    fft_power = np.abs(fft_vals) ** 2

    # Convert freq to period (months), skip DC component (freq=0)
    nonzero  = fft_freq > 0
    freqs_nz = fft_freq[nonzero]
    power_nz = fft_power[nonzero]
    periods  = 1.0 / freqs_nz        # period in months

    # Find top peaks in power spectrum
    peaks, props = find_peaks(power_nz, height=np.percentile(power_nz, 50))
    top3_peaks   = peaks[np.argsort(power_nz[peaks])[::-1][:3]]

    fig2, ax2 = plt.subplots(figsize=(14, 5))
    ax2.bar(periods, power_nz, width=0.3, color='#2196F3', alpha=0.7, label='Power')
    ax2.set_xscale('log')

    for rank, pk in enumerate(top3_peaks, 1):
        p = periods[pk]
        ax2.axvline(p, color=f'C{rank}', linestyle='--', linewidth=2)
        ax2.text(p * 1.05, max(power_nz) * (0.95 - rank * 0.12),
                 f'#{rank}: {p:.1f}m', color=f'C{rank}',
                 fontsize=9, fontweight='bold')

    # Mark known seasonal periods
    for sp, sname, sclr in [(3, '3m (Quarterly)', '#E91E63'),
                            (6, '6m (Semi-annual)', '#FF9800'),
                            (12, '12m (Annual)', '#4CAF50')]:
        if sp <= max(periods):
            ax2.axvline(sp, color=sclr, linestyle=':', linewidth=1.5, alpha=0.6)
            ax2.text(sp * 0.85, max(power_nz) * 0.50, sname,
                     color=sclr, fontsize=8, rotation=90, alpha=0.8)

    ax2.set_xlabel('Period (months, log scale)')
    ax2.set_ylabel('Power (magnitude²)')
    ax2.set_title(f'FFT Power Spectrum  —  {label}')
    ax2.legend()
    plt.tight_layout(); plt.show()

    if len(top3_peaks) > 0:
        print(f"\n  Top dominant FFT periods: "
              + ", ".join([f"{periods[p]:.1f}m" for p in top3_peaks]))

    # ══════════════════════════════════════════════════════════════════════
    # Panel 3  —  STL Decomposition
    # ══════════════════════════════════════════════════════════════════════
    # Use period=3 (quarterly) if n >= 2*period; else skip
    for stl_period, period_name in [(3, 'Quarterly (period=3)'),
                                    (6, 'Semi-annual (period=6)')]:
        if n < 2 * stl_period + 1:
            print(f"  [SKIP] STL period={stl_period}: not enough data points (n={n})")
            continue

        stl_result = STL(vals, period=stl_period, robust=True).fit()
        trend_c    = stl_result.trend
        seasonal_c = stl_result.seasonal
        residual_c = stl_result.resid

        # Seasonal strength index: 1 - Var(resid) / Var(seasonal + resid)
        var_resid = np.var(residual_c)
        var_seas  = np.var(seasonal_c + residual_c)
        strength  = max(0, 1 - var_resid / var_seas) if var_seas > 0 else 0

        fig3, axes3 = plt.subplots(4, 1, figsize=(18, 12), sharex=True)
        fig3.suptitle(
            f'STL Decomposition  ({period_name})  —  {label}\n'
            f'Seasonal Strength Index: {strength:.3f}  '
            f'{"(STRONG)" if strength > 0.6 else "(MODERATE)" if strength > 0.3 else "(WEAK)"}',
            fontsize=13, fontweight='bold'
        )

        components = [
            (vals,       'Observed', PALETTE['raw']),
            (trend_c,    'Trend',    PALETTE['ma3']),
            (seasonal_c, 'Seasonal', PALETTE['poly2']),
            (residual_c, 'Residual', '#607D8B'),
        ]
        for ax3, (comp, comp_name, comp_color) in zip(axes3, components):
            ax3.plot(dates, comp, color=comp_color, linewidth=2)
            if comp_name != 'Residual':
                ax3.fill_between(dates, comp, alpha=0.12, color=comp_color)
            else:
                ax3.axhline(0, color='black', linewidth=0.7, linestyle='--')
            ax3.axvline(pd.to_datetime(LAST_TRAIN_DATE), color='grey',
                        linestyle=':', linewidth=1.2)
            ax3.set_ylabel(comp_name, fontsize=10)
            ax3.yaxis.set_tick_params(labelsize=8)

        axes3[-1].set_xlabel('Date')
        axes3[-1].tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()
        print(f"  → Seasonal Strength Index (period={stl_period}): {strength:.4f}")
    print()


for series, label in [(train_series, '[PART A — TRAIN ONLY]'),
                      (full_series,  '[PART B — TRAIN + TEST]')]:
    plot_seasonality(series, label)

## 3.3  Cycle Detection

### Spectral Analysis (Periodogram)
The **Power Spectral Density (PSD)** from `scipy.signal.periodogram` estimates how the variance of the series is distributed across frequencies.  
Unlike FFT which shows amplitude at each frequency, the periodogram normalises by the sampling interval, giving a true **power density** (variance per frequency unit).  
Peaks in the PSD correspond to **dominant cycles** in the data.

### Continuous Wavelet Transform (CWT) — Scalogram
Classical spectral analysis assumes the cycle structure is **fixed over time** (stationary in frequency).  
The CWT splits signal energy across **both time and scale simultaneously**, revealing which cycles are active at which time periods.  
This is critical for short financial series where cycles may emerge or fade across the time horizon.

- **Wavelet**: Morlet (`'cmor1-1.5'`) — best for oscillatory signals
- **Scales → Periods**: period ≈ scale × wavelet centre frequency
- **Scalogram**: heatmap of |CWT coefficient|², warmer = stronger cycle at that time/period

In [ ]:
def plot_cycles(series, label):
    """
    Two-phase cycle detection:
      1. Periodogram (power spectral density) — which periods dominate
      2. CWT Scalogram (Morlet) — how those cycles evolve over time
    """
    vals  = series.values.astype(float)
    n     = len(vals)
    dates = pd.to_datetime(series.index)

    # Detrend by removing linear trend (better spectral clarity)
    t = np.arange(n)
    coefs     = np.polyfit(t, vals, 1)
    vals_detr = vals - np.polyval(coefs, t)

    # ══════════════════════════════════════════════════════════════════════
    # Panel 1  —  Periodogram (PSD)
    # ══════════════════════════════════════════════════════════════════════
    freqs, psd = periodogram(vals_detr, fs=1.0)   # fs=1 sample/month

    # Remove DC component (freq=0)
    nonzero    = freqs > 0
    freqs_nz   = freqs[nonzero]
    psd_nz     = psd[nonzero]
    periods_nz = 1.0 / freqs_nz

    # Detect peaks
    peaks, _     = find_peaks(psd_nz, height=np.percentile(psd_nz, 60))
    top_peaks    = peaks[np.argsort(psd_nz[peaks])[::-1][:5]]
    dom_periods  = periods_nz[top_peaks] if len(top_peaks) > 0 else []

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f'Spectral Analysis (Periodogram)  —  {label}', fontsize=13, fontweight='bold')

    # Left: PSD vs period
    ax1.semilogy(periods_nz, psd_nz, color='#2196F3', linewidth=1.5, label='PSD')
    ax1.fill_between(periods_nz, psd_nz, alpha=0.15, color='#2196F3')

    colors_cycle = plt.cm.Set1(np.linspace(0, 0.7, len(top_peaks)))
    for rank, (pk, clr) in enumerate(zip(top_peaks, colors_cycle), 1):
        p = periods_nz[pk]
        ax1.axvline(p, color=clr, linestyle='--', linewidth=2)
        ax1.text(p * 1.05, psd_nz[pk] * 1.5,
                 f'#{rank}: {p:.1f}m', color=clr, fontsize=9, fontweight='bold')

    for sp, sname, sclr in [(3, 'Q', '#E91E63'), (6, 'SA', '#FF9800')]:
        ax1.axvline(sp, color=sclr, linestyle=':', linewidth=1.5, alpha=0.6)
        ax1.text(sp, ax1.get_ylim()[0] * 1.5, sname, color=sclr, fontsize=9)

    ax1.set_xlabel('Period (months)')
    ax1.set_ylabel('Power Spectral Density (log)')
    ax1.set_title('PSD vs Period')
    ax1.legend()

    # Right: PSD vs frequency
    ax2.semilogy(freqs_nz, psd_nz, color='#4CAF50', linewidth=1.5)
    ax2.fill_between(freqs_nz, psd_nz, alpha=0.15, color='#4CAF50')
    for sp, sclr in [(3, '#E91E63'), (6, '#FF9800')]:
        ax2.axvline(1/sp, color=sclr, linestyle=':', linewidth=1.5, alpha=0.7)
        ax2.text(1/sp * 1.03, ax2.get_ylim()[0] * 1.5,
                 f'1/{sp}m', color=sclr, fontsize=8)
    ax2.set_xlabel('Frequency (cycles/month)')
    ax2.set_ylabel('Power Spectral Density (log)')
    ax2.set_title('PSD vs Frequency')

    plt.tight_layout()
    plt.show()

    print(f"\n  Dominant cycle periods detected: "
          + (", ".join([f"{p:.1f}m" for p in dom_periods]) if len(dom_periods) else "none"))

    # ══════════════════════════════════════════════════════════════════════
    # Panel 2  —  CWT Scalogram (Morlet wavelet)
    # ══════════════════════════════════════════════════════════════════════
    # Build scales mapping to periods 1→n/2 months
    wavelet       = 'cmor1-1.5'    # complex Morlet; bandwidth=1, center freq=1.5
    center_freq   = pywt.central_frequency(wavelet)
    # periods we want to probe (1 to n//2 months)
    target_periods = np.arange(1, n // 2 + 1, dtype=float)
    scales        = center_freq * n / target_periods   # scale = f_c * N / period (simplified)
    scales        = scales[scales > 0]

    # CWT computation
    cwt_coefs, cwt_freqs = pywt.cwt(vals_detr, scales, wavelet, sampling_period=1)
    cwt_periods           = 1.0 / cwt_freqs
    cwt_power             = np.abs(cwt_coefs) ** 2

    fig2, (ax3, ax4) = plt.subplots(2, 1, figsize=(18, 10),
                                     gridspec_kw={'height_ratios': [1, 4]})
    fig2.suptitle(f'Wavelet Scalogram (Morlet CWT)  —  {label}',
                  fontsize=13, fontweight='bold')

    # Top panel: raw series for time reference
    ax3.plot(np.arange(n), vals, color=PALETTE['raw'], linewidth=2)
    ax3.set_ylabel('Raw Series')
    ax3.set_xlim([0, n - 1])
    ax3.tick_params(labelbottom=False)
    ax3.axvline(sum(1 for d in series.index
                    if d <= datetime.date.fromisoformat(LAST_TRAIN_DATE)) - 1,
                color='grey', linestyle='--', linewidth=1.2, label='Train cutoff')
    ax3.legend(fontsize=8)

    # Bottom panel: scalogram heatmap
    im = ax4.contourf(np.arange(n), cwt_periods, cwt_power,
                      levels=64, cmap='viridis')
    cbar = plt.colorbar(im, ax=ax4, pad=0.02)
    cbar.set_label('Wavelet Power (|coef|²)', fontsize=9)

    # Mark dominant periods from periodogram as horizontal lines
    for rank, dp in enumerate(dom_periods[:3], 1):
        if dp <= cwt_periods.max():
            ax4.axhline(dp, color=f'C{rank}', linestyle='--',
                        linewidth=2, alpha=0.85,
                        label=f'Dominant cycle: {dp:.1f}m')

    # Train/Test cutoff vertical line
    cutoff_idx = sum(1 for d in series.index
                     if d <= datetime.date.fromisoformat(LAST_TRAIN_DATE)) - 1
    ax4.axvline(cutoff_idx, color='white', linestyle='--',
                linewidth=1.5, alpha=0.7, label='Train cutoff')

    # X-axis: convert integer index to readable dates
    tick_positions = np.linspace(0, n - 1, min(n, 12), dtype=int)
    ax4.set_xticks(tick_positions)
    ax4.set_xticklabels([str(series.index[i]) for i in tick_positions], rotation=45)
    ax4.set_ylabel('Period (months)')
    ax4.set_xlabel('Date')
    ax4.set_ylim([1, min(12, cwt_periods.max())])
    ax4.set_title('Warmer colour = stronger cycle at that time & period')
    ax4.legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()


for series, label in [(train_series, '[PART A — TRAIN ONLY]'),
                      (full_series,  '[PART B — TRAIN + TEST]')]:
    plot_cycles(series, label)

# Section 4 — Summary Dashboard

A single consolidated figure combining the four most informative views on one shared time axis:
1. **Raw Series** — the observed data
2. **STL Seasonal Component** — extracted periodicity
3. **FFT Power Spectrum** — frequency domain view
4. **Wavelet Scalogram** — time-frequency cycle map

In [ ]:
def summary_dashboard(series, label):
    """
    4-panel summary dashboard for a given series.
    Panels share the time (x) axis where applicable.
    """
    vals    = series.values.astype(float)
    n       = len(vals)
    dates   = pd.to_datetime(series.index)
    t       = np.arange(n)

    # ── Pre-compute components ────────────────────────────────────────────
    # Detrended values for spectral/wavelet
    coefs      = np.polyfit(t, vals, 1)
    vals_detr  = vals - np.polyval(coefs, t)

    # STL seasonal component (period=3 if enough data, else period=2)
    stl_period = 3 if n >= 7 else 2
    stl_result = STL(vals, period=stl_period, robust=True).fit()
    seasonal_c = stl_result.seasonal

    # FFT
    fft_vals   = rfft(vals_detr)
    fft_freq   = rfftfreq(n, d=1)
    fft_power  = np.abs(fft_vals) ** 2
    nonzero    = fft_freq > 0
    freqs_nz   = fft_freq[nonzero]
    power_nz   = fft_power[nonzero]
    periods_nz = 1.0 / freqs_nz

    # CWT scalogram
    wavelet     = 'cmor1-1.5'
    center_freq = pywt.central_frequency(wavelet)
    tgt_periods = np.arange(1, n // 2 + 1, dtype=float)
    scales      = center_freq * n / tgt_periods
    scales      = scales[scales > 0]
    cwt_coefs, cwt_freqs = pywt.cwt(vals_detr, scales, wavelet, sampling_period=1)
    cwt_periods = 1.0 / cwt_freqs
    cwt_power   = np.abs(cwt_coefs) ** 2

    # ── Figure layout ─────────────────────────────────────────────────────
    fig = plt.figure(figsize=(20, 16))
    gs  = gridspec.GridSpec(4, 2,
                            height_ratios=[2, 2, 2, 3],
                            hspace=0.45, wspace=0.3)

    ax_raw      = fig.add_subplot(gs[0, :])      # full width — raw series
    ax_seasonal = fig.add_subplot(gs[1, :])      # full width — seasonal
    ax_fft      = fig.add_subplot(gs[2, 0])      # half — FFT
    ax_fft_ann  = fig.add_subplot(gs[2, 1])      # half — FFT annotations
    ax_wavelet  = fig.add_subplot(gs[3, :])      # full width — scalogram

    fig.suptitle(
        f'Decomposition Dashboard  —  {TARGET_METRIC}\n'
        f'{PORTFOLIO} / {SUB_PORTFOLIO}  ·  {label}',
        fontsize=15, fontweight='bold', y=0.98
    )

    # ── 1. Raw series ─────────────────────────────────────────────────────
    ax_raw.plot(dates, vals, color=PALETTE['raw'], linewidth=2,
                marker='o', markersize=4, label='Actual')
    ax_raw.plot(dates, stl_result.trend, color=PALETTE['ma3'],
                linewidth=2.5, linestyle='--', label='STL Trend')
    ax_raw.axvline(pd.to_datetime(LAST_TRAIN_DATE), color='grey',
                   linestyle='--', linewidth=1.2, label='Train cutoff')
    ax_raw.fill_between(dates, vals, alpha=0.1, color=PALETTE['raw'])
    ax_raw.set_ylabel('Outstanding ($)'); ax_raw.legend(ncol=3)
    ax_raw.set_title('① Observed Series + Trend')
    ax_raw.tick_params(axis='x', rotation=30)

    # ── 2. Seasonal component ─────────────────────────────────────────────
    ax_seasonal.bar(dates, seasonal_c, color=PALETTE['poly2'], alpha=0.7,
                    width=18, label=f'Seasonal (STL period={stl_period})')
    ax_seasonal.axhline(0, color='black', linewidth=0.8)
    ax_seasonal.axvline(pd.to_datetime(LAST_TRAIN_DATE), color='grey',
                        linestyle='--', linewidth=1.2)
    ax_seasonal.set_ylabel('Seasonal Effect'); ax_seasonal.legend()
    ax_seasonal.set_title('② Seasonal Component (STL)')
    ax_seasonal.tick_params(axis='x', rotation=30)

    # ── 3. FFT Power Spectrum ─────────────────────────────────────────────
    ax_fft.semilogy(periods_nz, power_nz, color='#2196F3', linewidth=1.5)
    ax_fft.fill_between(periods_nz, power_nz, alpha=0.15, color='#2196F3')
    peaks_p, _ = find_peaks(power_nz, height=np.percentile(power_nz, 50))
    top_p      = peaks_p[np.argsort(power_nz[peaks_p])[::-1][:3]]
    for rank, pk in enumerate(top_p, 1):
        p = periods_nz[pk]
        ax_fft.axvline(p, color=f'C{rank}', linestyle='--', linewidth=2)
        ax_fft.text(p * 1.05, power_nz[pk] * 2,
                    f'{p:.1f}m', color=f'C{rank}', fontsize=8, fontweight='bold')
    ax_fft.set_xlabel('Period (months)')
    ax_fft.set_ylabel('Power (log)')
    ax_fft.set_title('③ FFT Power Spectrum')

    # Annotation panel (replace with text summary)
    ax_fft_ann.axis('off')
    peaks_text = "\n".join([f"  #{r}: {periods_nz[pk]:.1f} months"
                            for r, pk in enumerate(top_p, 1)]) \
                 if len(top_p) else "  No clear peaks detected"

    # Hurst (reuse if available)
    hurst_str = "N/A (< 12 pts)" if n < 12 else ""
    if n >= 12:
        H, _, _ = compute_Hc(vals, kind='price', simplified=True)
        hurst_str = f"{H:.3f}  ({'Long memory' if H > 0.55 else 'Random Walk' if H > 0.45 else 'Mean-reverting'})"

    # ADF
    _, adf_p, *_ = adfuller(vals, autolag='AIC')
    adf_str = f"{'Stationary' if adf_p < 0.05 else 'Non-stationary'}  (p={adf_p:.3f})"

    ann_text = (
        f"  Series Summary\n"
        f"  {'─' * 30}\n"
        f"  Data points : {n}\n"
        f"  Date range  : {series.index[0]} → {series.index[-1]}\n"
        f"  Mean        : {vals.mean():,.0f}\n"
        f"  Std         : {vals.std():,.0f}\n\n"
        f"  ADF test    : {adf_str}\n"
        f"  Hurst (H)   : {hurst_str}\n\n"
        f"  Top FFT periods:\n{peaks_text}"
    )
    ax_fft_ann.text(0.05, 0.95, ann_text, transform=ax_fft_ann.transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.8))
    ax_fft_ann.set_title('Summary Statistics')

    # ── 4. Wavelet Scalogram ──────────────────────────────────────────────
    im = ax_wavelet.contourf(np.arange(n), cwt_periods, cwt_power,
                             levels=64, cmap='viridis')
    cbar = plt.colorbar(im, ax=ax_wavelet, pad=0.01, fraction=0.03)
    cbar.set_label('Wavelet Power', fontsize=9)

    # Dominant period lines from periodogram
    for rank, pk in enumerate(top_p[:3], 1):
        dp = periods_nz[pk]
        if dp <= cwt_periods.max():
            ax_wavelet.axhline(dp, color=f'C{rank}', linestyle='--',
                               linewidth=2, alpha=0.9,
                               label=f'Dominant: {dp:.1f}m')
    cutoff_idx = sum(1 for d in series.index
                     if d <= datetime.date.fromisoformat(LAST_TRAIN_DATE)) - 1
    ax_wavelet.axvline(cutoff_idx, color='white', linestyle='--',
                       linewidth=2, alpha=0.75, label='Train cutoff')

    tick_pos = np.linspace(0, n - 1, min(n, 14), dtype=int)
    ax_wavelet.set_xticks(tick_pos)
    ax_wavelet.set_xticklabels([str(series.index[i]) for i in tick_pos], rotation=45)
    ax_wavelet.set_ylabel('Period (months)')
    ax_wavelet.set_xlabel('Date')
    ax_wavelet.set_ylim([1, min(10, cwt_periods.max())])
    ax_wavelet.set_title('④ Wavelet Scalogram (Morlet CWT) — Time × Period')
    ax_wavelet.legend(loc='upper right', fontsize=8)

    plt.show()


# ── Run dashboard for both parts ──────────────────────────────────────────
for series, label in [(train_series, '[PART A — TRAIN ONLY]'),
                      (full_series,  '[PART B — TRAIN + TEST]')]:
    summary_dashboard(series, label)